# Infererence.py를 수정하여 매개변수로 --use-peft를 입력받아 직접 Peft 어댑터 모델을 로드할 수 있도록 수정되었다.

FLASK를 이용하여 벤치마킹을 하기에 앞서서, 먼저 연구진들이 미리 제작한 질문지를 목표 모델에게 추론시키는 작업이 필요하다. 당연한 말이지만 해당 모델이 내놓는 결과물을 분석하여 성능을 벤치마킹하기 때문이다.
1. 구글 드라이브를 마운트하여 FLASK 폴더에 접근한다.
2. FLASK 프레임워크에서 사용되는 ray 패키지를 설치한다.
3. 추론 작업을 실행하는 파이썬 파일을 실행한다.
4. 필요에 따라서 --use-peft 매개변수를 True로 전달한다.

이 테스트 노트북에서는 10개의 질문만을 처리하여 정상작동을 확인했다.
T4 GPU 기준으로 1700개 질문을 처리하는데 15시간 정도 걸린다.
안타깝게도 파일 디렉토리만 적는 실수를 하여 15시간이나 써서 얻은 데이터는 날라갔다.
**본 작업을 진행하기 전에 정상적으로 저장되는지 확인부터 하기 바란다.**

* --model-path : 대상 모델의 허깅페이스 경로
* --model-id : 모델 아이디인데 어떤 목적으로 사용되는 지 모르겠다
* --question-file : 모델의 입력에 사용되는 질문 파일의 위치와 이름
* --answer-file : 모델의 출력이 저장되는 대답 파일의 위치와 이름
* --num-gpus : 사용되는 gpu의 개수이며 1로 설정하면 된다
* --use-peft : 대상 모델이 peft 모델이라면 True로 설정한다


In [ ]:
!nvidia-smi

Sun Dec  1 04:19:16 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA L4                      Off | 00000000:00:03.0 Off |                    0 |
| N/A   37C    P8              12W /  72W |      1MiB / 23034MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [ ]:
#그래픽카드 설정
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
#구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install ray

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.3/66.3 MB 31.3 MB/s eta 0:00:00


허깅페이스에서 로드할 생각이라면 로그인 과정이 필요하다.

In [ ]:
from google.colab import userdata
read_token = userdata.get('huggingface_read')
!huggingface-cli login --token $read_token

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: read).
The token `Read` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `Read`


* --model-path를 작성할 때에 구글드라이브에 저장된 모델 혹은 허깅페이스에서 로드할 수 있다. 구글드라이브에서 로드할 생각이라면 경로를 입력하면 되고, 아니라면 허깅페이스 리포지토리의 주소를 적으면 된다.
* --question-file은 미리 만들어놓은 10개짜리 평가용 파일을 적었다.
* --answer-file은 추론 결과물이 저장될 경로를 지정하면 된다.

In [ ]:
!pip install jsonlines

In [ ]:
import jsonlines
def create_empty_jsonl_file(file_path):
    # 빈 데이터
    data = []

    # 해당 경로에 파일이 존재하는지 확인
    if os.path.exists(file_path):
      print(f"Error: file '{file_path}' exists.")
    else:
        # 해당 경로에 빈 데이터를 가진 파일 생성
        with jsonlines.open(file_path, mode='w') as writer:
            writer.write_all(data)
        print(f"Empty JSONL file created at: {file_path}")

def create_empty_directory(directory_path):
    if not os.path.exists(directory_path):
        os.makedirs(directory_path)
        print(f"Empty Directory created at: {directory_path}")
    else:
        print(f"Error: file '{directory_path}' exists.")

# Peft 모델 추론 시


In [ ]:
#====================사용자의 입력 필요====================
#Peft 어댑터 모델의 허깅페이스 리포지토리
model_path = "prayslaks/mistral_flask_evaluation_2000_fedavg_c5s3_i10_b4a1_l2048_r16a32_20241129133706"

#모델의 저장 이름
model_1 = "Mistral_FlaskDataset_c5s3"
model_2 = "Prometheus_FlaskDataset_c5s3"
save_model_name = model_1

#모델의 체크포인트
iteration = 1

#질문의 개수
question_num = 1000
#==========================================================

#====================Peft 어댑터 모델=====================
#Peft 어댑터 모델 추론 파이썬 코드
inference_code = "/content/drive/MyDrive/OpenFedFLASK/FLASK/model_output/inference_peft.py"

#Peft 어댑터 모델의 서브 폴더
peft_subfolder = f"checkpoint-{iteration}"

#입력에 사용할 질문 파일
question_file = f"/content/drive/MyDrive/OpenFedFLASK/FLASK/input_data/flask_evaluation_q{question_num}.jsonl"

#추론 응답이 저장되는 경로
answer_directory = f"/content/drive/MyDrive/OpenFedFLASK/flask_inference_outputs/{save_model_name}"

#추론 응답이 저장되는 파일
answer_file = f"/content/drive/MyDrive/OpenFedFLASK/flask_inference_outputs/{save_model_name}/q{question_num}_i{iteration}.jsonl"
#==========================================================

In [ ]:
create_empty_directory(answer_directory)
create_empty_jsonl_file(answer_file)

Error: file '/content/drive/MyDrive/OpenFedFLASK/flask_inference_outputs/Mistral_FlaskDataset' exists.
Empty JSONL file created at: /content/drive/MyDrive/OpenFedFLASK/flask_inference_outputs/Mistral_FlaskDataset/q1000_i1.jsonl


In [ ]:
!python $inference_code --model-path $model_path --model-id alpaca --question-file $question_file --answer-file $answer_file --num-gpus 1 --peft-subfolder $peft_subfolder --use-peft True

Streaming output truncated to the last 5000 lines.
(get_model_answers pid=7379) 
(get_model_answers pid=7379) MONTAGE OF THE SCENE:
(get_model_answers pid=7379) 
(get_model_answers pid=7379) - A group of women in bikinis play beach volleyball.
(get_model_answers pid=7379) - A group of older men in Speedos play poker.
(get_model_answers pid=7379) - A group of young men play ping pong.
(get_model_answers pid=7379) - A group of young women sunbathe on the beach.
(get_model_answers pid=7379) - A group of old people sunbathe on the beach.
(get_model_answers pid=7379) 
(get_model_answers pid=7379) INT. BAR - DAY
(get_model_answers pid=7379) 
(get_model_answers pid=7379) JdeG (30s, male, with a lanky, muscular build, wearing a white t-shirt, black shorts and sunglasses), behind the bar.
(get_model_answers pid=7379) 
(get_model_answers pid=7379) SOPHIE SNazz (20s, female, with a tanned, athletic build, wearing a white bikini top and denim shorts), behind the bar next to JdeG.
(get_model_answer

# 일반 모델 추론 시

In [ ]:
#====================사용자의 입력 필요====================
#일반 모델의 허깅페이스 리포지토리
model_path = "mistralai/Mistral-7B-Instruct-v0.3"

#모델의 저장 이름
model_1 = "Mistral_FlaskDataset"
model_2 = "Prometheus_FlaskDataset"
save_model_name = model_1

#질문의 개수
question_num = 1000
#==========================================================

#====================일반 모델=====================
#일반 모델 추론 파이썬 코드
inference_code = "/content/drive/MyDrive/OpenFedFLASK/FLASK/model_output/inference.py"

#모델의 체크포인트
iteration = 0

#입력에 사용할 질문 파일
question_file = f"/content/drive/MyDrive/OpenFedFLASK/FLASK/input_data/flask_evaluation_q{question_num}.jsonl"

#추론 응답이 저장되는 파일
answer_file = f"/content/drive/MyDrive/OpenFedFLASK/flask_inference_outputs/{save_model_name}/q{question_num}_i{iteration}.jsonl"
#==========================================================

In [ ]:
create_empty_jsonl_file(answer_file)

Empty JSONL file created at: /content/drive/MyDrive/OpenFedFLASK/flask_inference_outputs/Mistral_FlaskDataset/q1000_i0.jsonl


In [ ]:
!python $inference_code --model-path $model_path --model-id alpaca --question-file $question_file --answer-file $answer_file --num-gpus 1